### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [1]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 75.1 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 123.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 22.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 MB 56.8 MB/s  0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.19.9-py3-none-any.whl size=2192410 sha256=6749c6925ed47de925d4ced31bf009d7c4781ae919cb8156dd97db3df3436bae
  Stored in directory: /root/.cache/pip/wheels/99/42/13/14d3508ae75d66e803dff6d82342500415196432f3e10cafe1
Successfully built deepspeed
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datase

### Load data and model

In [2]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


train.jsonl: reconstructing file:   0%|          |  0.00B / 70.8MB            

train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/7.83M [00:00<?, ?B/s]

test.jsonl: reconstructing file:   0%|          |  0.00B / 76.0MB            

test.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [3]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  433MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

### Tokenize the data

In [4]:
MAX_LENGTH = 128
def preprocess_function(examples):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(preprocess_function, batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  433MB            

model.safetensors: downloading bytes:           |  0.00B            

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [5]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [6]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [7]:
import torch
import torch.nn.functional as F

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

model.train()

for batch in train_loader:
    optimizer.zero_grad()

    outputs = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids']
    )

    loss = F.cross_entropy(outputs.logits, batch['labels'])

    loss.backward()
    optimizer.step()

model.eval()

for batch in val_loader:
    break

print("Sample batch:", batch)

with torch.no_grad():
    predicted = model(
        input_ids=batch['input_ids'],
        attention_mask=batch['attention_mask'],
        token_type_ids=batch['token_type_ids']
    )

print('\nPrediction (probs):',
      torch.softmax(predicted.logits, dim=1).cpu().numpy())

Sample batch: {'labels': tensor([0]), 'idx': tensor([0]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,   

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [13]:
import torch
from torch.utils.data import DataLoader
from transformers import default_data_collator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

val_loader = DataLoader(
    val_loader.dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=(device.type == "cuda"),
    collate_fn=default_data_collator
)

In [14]:
correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        inputs = {
            key: batch[key].to(device)
            for key in ("input_ids", "attention_mask", "token_type_ids")
            if key in batch
        }

        outputs = model(**inputs)
        predictions = outputs.logits.argmax(dim=-1)

        labels = batch["labels"].to(device)

        correct += (predictions == labels).sum().item()
        total += labels.numel()

accuracy = correct / total

print(f"Validation accuracy: {accuracy:.4f}")

Validation accuracy: 0.9084


In [15]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://huggingface.co/docs/transformers/en/tasks/sequence_classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [17]:
import time
import gc
import torch
import pandas as pd

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding
)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print("Device:", device)

model_names = {
    "BERT (baseline)": "textattack/bert-base-uncased-SST-2",
    "DistilBERT": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    "RoBERTa": "textattack/roberta-base-SST-2",
    "ALBERT": "textattack/albert-base-v2-SST-2"
}

dataset = load_dataset("nyu-mll/glue", "sst2", split="validation")

batch_size = 64
max_length = 128

results = []

for name, model_id in model_names.items():
    print(f"\nEvaluating {name}...")

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    current_model = AutoModelForSequenceClassification.from_pretrained(
        model_id
    ).to(device)
    current_model.eval()

    def tokenize(examples):
        return tokenizer(
            examples["sentence"],
            truncation=True,
            max_length=max_length
        )

    tokenized = dataset.map(
        tokenize,
        batched=True,
        remove_columns=dataset.column_names
    )

    # Add ground-truth labels back
    tokenized = tokenized.add_column(
        "labels", dataset["label"]
    )

    loader = DataLoader(
        tokenized,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=(device.type == "cuda"),
        collate_fn=DataCollatorWithPadding(tokenizer)
    )

    correct = 0
    total = 0

    if device.type == "cuda":
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.inference_mode():
        for batch in loader:
            labels = batch.pop("labels").to(device)

            inputs = {
                k: v.to(device)
                for k, v in batch.items()
            }

            logits = current_model(**inputs).logits
            predictions = logits.argmax(dim=-1)

            correct += (predictions == labels).sum().item()
            total += labels.numel()

    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()

    elapsed = time.perf_counter() - start

    accuracy = correct / total
    speed = total / elapsed

    size_mb = sum(
        p.numel() * p.element_size()
        for p in current_model.parameters()
    ) / (1024 ** 2)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Speed: {speed:.2f} samples/sec")
    print(f"Size: {size_mb:.2f} MB")

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Speed (samples/sec)": speed,
        "Size (MB)": size_mb
    })

    del current_model, loader
    gc.collect()

    if device.type == "cuda":
        torch.cuda.empty_cache()

df = pd.DataFrame(results)

print("\nFinal comparison:")
print(df.to_string(index=False))

df.to_csv("model_comparison.csv", index=False)

Device: cuda


README.md:   0%|          | 0.00/35.3k [00:00<?, ?B/s]

sst2/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.11MB            

sst2/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sst2/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 72.8kB            

sst2/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (43 outputs left out)

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 46.7MB            

model.safetensors: downloading bytes:           |  0.00B            

Accuracy: 0.9243
Speed: 279.88 samples/sec
Size: 44.58 MB

Final comparison:
          Model  Accuracy  Speed (samples/sec)  Size (MB)
BERT (baseline)  0.924312           330.702963 417.647469
     DistilBERT  0.910550           660.915391 255.413094
        RoBERTa  0.940367           337.306916 475.491219
         ALBERT  0.924312           279.875634  44.575203


In [19]:
best_accuracy = df.loc[df["Accuracy"].idxmax()]
fastest = df.loc[df["Speed (samples/sec)"].idxmax()]
smallest = df.loc[df["Size (MB)"].idxmin()]

report = f"""
Model Comparison Report

We evaluated four transformer-based models on the GLUE SST-2
validation dataset using the same hardware ({device}),
batch size ({batch_size}), and maximum sequence length ({max_length}).

The highest validation accuracy was achieved by
{best_accuracy['Model']} with an accuracy of
{best_accuracy['Accuracy']:.4f}.

The fastest model was {fastest['Model']}, processing
{fastest['Speed (samples/sec)']:.2f} samples per second.

The smallest model was {smallest['Model']}, requiring
{smallest['Size (MB)']:.2f} MiB for its parameters.
"""

print(report)


Model Comparison Report

We evaluated four transformer-based models on the GLUE SST-2
validation dataset using the same hardware (cuda),
batch size (64), and maximum sequence length (128).

The highest validation accuracy was achieved by
RoBERTa with an accuracy of
0.9404.

The fastest model was DistilBERT, processing
660.92 samples per second.

The smallest model was ALBERT, requiring
44.58 MiB for its parameters.



### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.